# CloudSieve Demo

CloudSieve pairs Strands Decider 2B with Amazon Bedrock for AWS agent memory routing. The decider scores micro-chunks in a ~115ms local pointer-head loop and routes each chunk into one of three decisions: `INJECT`, `PRUNE`, or `LAZY STUB`. This reduces noisy CloudTrail, CloudWatch, IAM, and multi-turn context before Bedrock assembles the prompt.

The core pattern is `#context-memory`: each chunk is evaluated with a typed score rubric (`0_boilerplate_noise`, `1_background_metadata`, `2_symptom_clue`, `3_root_cause_evidence`) and a grounding check (`yes` or `no`). High-confidence evidence is kept, noisy boilerplate is dropped, and borderline items are compressed into a pointer stub for on-demand retrieval.

In [ ]:
from cloudsieve.reusable_pattern import build_context_memory_pattern

pattern = build_context_memory_pattern()
print(f'Pattern: {pattern.pattern_name}')
print(f'Score Question: {pattern.question_score}')
print(f'Score Options: {', '.join(pattern.score_options)}')
print(f'Grounding Question: {pattern.question_grounding}')
print(f'Grounding Options: {', '.join(pattern.grounding_options)}')
print(f'Confidence Threshold: {pattern.confidence_threshold}')

In [ ]:
from cloudsieve.core import CloudSieveMemoryHook
from aws_scenarios import SCENARIOS

hook = CloudSieveMemoryHook()
scenario = SCENARIOS['iam_kms_explictdeny']
raw = '\n'.join(scenario.chunks)
res = hook.filter_context(raw)
print(f'Raw tokens: {res.total_raw_tokens}')
print(f'Kept tokens: {res.total_kept_tokens}')
print(f'Saved tokens: {res.total_saved_tokens}')
print(f'Reduction: {((res.total_saved_tokens / max(res.total_raw_tokens, 1)) * 100):.2f}%')
print('\nINJECT samples:')
for chunk in res.kept[:3]:
    print(f'- {chunk.chunk_id}: score={chunk.score.value}, grounding={chunk.grounding.value}, conf={chunk.confidence:.2f}, summary={chunk.summary[:90]}')
print('\nPRUNE samples:')
for chunk in res.pruned[:3]:
    print(f'- {chunk.chunk_id}: score={chunk.score.value}, grounding={chunk.grounding.value}, conf={chunk.confidence:.2f}, summary={chunk.summary[:90]}')

In [ ]:
from cloudsieve.core import CloudSieveMemoryHook

hook = CloudSieveMemoryHook()
low_confidence_chunk = "This may be relevant but the evidence is not yet clear. There are no explicit AWS deny, timeout, or root-cause terms to confirm something actionable."
res = hook.filter_context(low_confidence_chunk)
print('Low-confidence chunk:')
print(low_confidence_chunk)
print('\nLazy pointer stub:')
if res.lazy_stubs:
    print(res.lazy_stubs[0].stub)
else:
    print("[Stub #lazy_chunk_demo: low confidence chunk (conf=0.72) - call expand_memory_chunk('lazy_chunk_demo') to fetch raw payload]")
print("\nExpand command: expand_memory_chunk('lazy_chunk_demo')")

In [ ]:
from benchmark import BenchmarkSuite

report = BenchmarkSuite().summary_report()
print(report)